## Load and prepare data (zarr)

Load and prepare data for backtrajectory calculations.

- Load the NSIDC PP data: `ds`

    - Loading a zarr array for quick access
    - Interpolating over a few holes in the data.
    
- Load a land mask: `MASK`
    - Define a land mask of land/no land that we use to kill trajectories that intersect a coastline.

In [1]:
import numpy as np
import xarray as xr
import glob
import pyproj
import matplotlib.pyplot as plt
from matplotlib.dates import date2num
import cmocean
import re

### Load data

Replace the *ddir* folder with the directory containing the PP NSIDC files (e.g. *icemotion_daily_nh_25km_20120101_20121231_v4.1.nc*)

Set year bounds; exclude years you are sure not to use (we will load the whole thing in memory, so good to be mindful of size - but 2014-2021 has been totally fine for me).

In [5]:
ddir = (
    '/media/oyvindf/SKADI/sea_ice/sea_ice_drift/'
    'nsidc_polar_pathfinder_v4/products/'
    'PPv4_drift_preprocesses_1978_2024.zarr/')

Lazy load of the data 

In [6]:
ds = xr.open_zarr(ddir, chunks={"time": -1, "y": 50, "x": 50})

### Define a land mask

Here, we can define a land mask (*MASK*) that will be used to kill a backtrajectory when it intersects a coastline.

*MASK* should have the following variables:
- *x, y* - Grid 
- *is_land* - Boolean (land=1, not land = 0)  
- *proj* - pyproj.CRS defining the grid projection of *x, y*


In my case, I am using a 2km land mask for the Eurasian sector based on IBCAO-v4, where I have removed Svalbard in order to allow trajectories to pass through islands. This mask was made elsewhere - I am just loading it here.

We can choose not to use a land mask. In that case, set *no_land_mask = True*

In [7]:
no_land_mask = False

In [8]:
# Load the land mask based on IBCAO (and defining the projection of this mask):
if not no_land_mask:
    my_mask_proj = pyproj.CRS('epsg:3996')  
    my_mask = xr.open_dataset(
        '/media/oyvindf/SKADI/geodata/bathymetry/ibcao_v4/netcdf/IBCAO_landmask_greater_barents_2km.nc')
    mask_is_land = my_mask.landmask_wo_svb
    # (Also adding lon/lat, but thise aren't strictly necessary)
    MASK = xr.Dataset(coords = {'x':my_mask.x, 'y':my_mask.y},
                     data_vars = {'lon':my_mask.lon, 'lat':my_mask.lat,
                                  'is_land':mask_is_land})
    MASK['proj'] = ((), my_mask_proj)

In [9]:
# If we dont have a land mask/dont want to use one:
if no_land_mask:
    MASK = None

##### (Optional): Quick check of mask

Toggle *show_mask=True* to show a quick landmask plot

In [10]:
show_mask = False

if show_mask:

    import cartopy.crs as ccrs
    import cartopy.feature as cfeature

    lat_ctr, lon_ctr = 86, 30

    cart_proj = ccrs.Stereographic(
        central_latitude=lat_ctr,
        central_longitude=lon_ctr,
        scale_factor=None, globe=None)

    fig = plt.figure(figsize = (8, 8))
    ax = plt.axes(projection=cart_proj)

    ax.set_facecolor('gray')
    # Coastline
    ax.coastlines('50m', color = 'w', lw = 0.5 )

    # Land
    land_50m = cfeature.NaturalEarthFeature(
        'physical', 'land', '50m',
        edgecolor='k', linewidth = 1,
        facecolor=cfeature.COLORS['land'])

   # ax.add_feature(land_50m)

    # Set boundaries
    ax.set_extent([-20, 191, 66, 89.5], crs=ccrs.PlateCarree())
    
    # Transform mask coordiates (x-y) to (lat-lon) 
  #  wgs84_proj = pyproj.CRS("EPSG:4326") # latlon with WGS84 datum
  #  MASK_to_latlon = pyproj.Transformer.from_crs(MASK.proj.data.item(), wgs84_proj)
  #  mask_lon, mask_lat = None, None

    ax.pcolormesh(MASK.x, MASK.y, MASK.is_land, 
                  cmap = cmocean.cm.haline, transform = ccrs.epsg(3996))
    